In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import scipy.stats

In [ ]:
from google.colab import files
upload = files.upload()

Saving atlantic_tech_mock_social_listening_raw.csv to atlantic_tech_mock_social_listening_raw (2).csv


In [ ]:
data = pd.read_csv('atlantic_tech_mock_social_listening_raw.csv')
data.columns
#data.head()

Index(['post_id', 'date', 'platform', 'country', 'audience_group',
       'source_type', 'post_text', 'narrative_theme', 'sentiment_label',
       'engagements', 'reach_estimate', 'misinformation_likelihood',
       'campaign_phase', 'recommended_asset', 'notes'],
      dtype='object')

In [ ]:
data.head()

,post_id,date,platform,country,audience_group,source_type,post_text,narrative_theme,sentiment_label,engagements,reach_estimate,misinformation_likelihood,campaign_phase,recommended_asset,notes
0,AT-10000,21/05/2026,X,NaN,NaN,Organic Post,Regulatory compliance cannot be an afterthough...,regulatory_compliance,negative,7.0,1089,0.05,trust correction,Compliance proof-point carousel,duplicate check
1,AT-10001,12/06/2026,YouTube,U.K.,Government buyer,press mention,Where is the line between emergency response a...,dual_use_ethics,negative,231.0,4586,0.94,trust correction,Civil defence ethics FAQ,NaN
2,AT-10002,30/05/2026,X,NL,Policy adviser,influencer post,New AI rules mean vendors need stronger transp...,regulatory_compliance,neutral,78.0,927,0.12,pre-launch research,Compliance proof-point carousel,NaN
3,AT-10003,21/05/2026,LinkedIn,FR,Public,forum thread,Cyber resilience claims need independent audit...,cyber_resilience,negative,268.0,5663,0.89,conversion,Security whitepaper,NaN
4,AT-10004,04-24-2026,LinkedIn,U.K.,Emergency services,NaN,Dual-use technology always gets sold as safety...,dual_use_ethics,negative,137.0,3079,0.58,pre-launch research,Civil defence ethics FAQ,NaN


1) Data Cleaning, pre-processing


In [ ]:
data.shape

(757, 15)

In [ ]:
clean = data.copy()
clean['date'] = pd.to_datetime(clean['date'],errors="coerce", dayfirst = True)
clean['date'] = clean['date']#.dt.date
clean.shape

(757, 15)

In [ ]:
clean.tail()

,post_id,date,platform,country,audience_group,source_type,post_text,narrative_theme,sentiment_label,engagements,reach_estimate,misinformation_likelihood,campaign_phase,recommended_asset,notes
752,AT-10142,NaT,trade media,DE,Public,NaN,Transparent AI for critical infrastructure cou...,positive_innovation,neutral,142.0,2520,NaN,pre-launch research,Launch film / case study,possible duplicate
753,AT-10026,2026-05-08,x,DE,Cybersecurity professional,opinion piece,Where is the line between emergency response a...,dual_use_ethics,neutral,50.0,1545,0.69,trust correction,Civil defence ethics FAQ,possible duplicate
754,AT-10731,2026-05-02,LinkedIn,DE,Public,opinion piece,Another AI vendor saying trust us while public...,data_sovereignty,neutral,69.0,1768,0.47,trust correction,Sovereign data explainer,possible duplicate
755,AT-99901,NaT,X,UK,Investor,comment,???,unknown,mixed,-5.0,lots,1.20,awareness,NaN,intentional anomaly
756,NaN,2026-06-05,LinkedIn,U.K.,Government buyer,organic post,Faster procurement should not mean bypassing s...,procurement trust,negative,84.0,2450,0.67,trust correction,Transparent procurement one-pager,missing id


In [ ]:
uk_map = {
    "U.K." : "UK",
    "UK" : "UK",
    "UNITED KINGDOM" : "UK",
    "ENGLAND" : "UK",
    "GB" : "UK",
    "GREAT BRITAIN" : "UK"
}

clean = clean.drop_duplicates()
clean = clean.dropna(subset=["date","platform","country", "narrative_theme", "sentiment_label"])

clean["country"] = clean["country"].astype(str).str.strip().str.upper()
clean["country"] = clean["country"].replace(uk_map)
clean = clean.dropna(subset=["date"])

clean_uk = clean[clean["country"] == "UK"].copy()
clean_uk['Calendar_Month']=clean_uk['date'].dt.month
clean_uk['Calendar_Year']=clean_uk['date'].dt.year
clean_uk['date'] = clean_uk['date']



In [ ]:
clean_uk.shape

(127, 17)

In [ ]:
clean_uk

,post_id,date,platform,country,audience_group,source_type,post_text,narrative_theme,sentiment_label,engagements,reach_estimate,misinformation_likelihood,campaign_phase,recommended_asset,notes,Calendar_Month,Calendar_Year
1,AT-10001,2026-06-12,YouTube,UK,Government buyer,press mention,Where is the line between emergency response a...,dual_use_ethics,negative,231.0,4586,0.94,trust correction,Civil defence ethics FAQ,NaN,6,2026
14,AT-10014,2026-05-28,Reddit,UK,Policy adviser,video comment,"Data sovereignty is the real issue here, not t...",data_sovereignty,neutral,91.0,1283,0.89,retargeting,Sovereign data explainer,NaN,5,2026
24,AT-10024,2026-03-28,YT,UK,Emergency services,forum thread,Cyber resilience claims need independent audit...,cyber_resilience,negative,23.0,611,0.81,pre-launch research,Security whitepaper,NaN,3,2026
44,AT-10044,2026-04-29,Forum,UK,Government buyer,opinion piece,HAS ATLANTIC EXPLAINED HOW IT WILL COMPLY WITH...,regulatory_compliance,neutral,19.0,224,0.09,pre-launch research,Compliance proof-point carousel,NaN,4,2026
45,AT-10045,2026-04-05,YouTube,UK,Civil society,video comment,This needs clear ethical boundaries before lau...,dual_use_ethics,negative,60.0,3914,0.87,trust correction,NaN,NaN,4,2026
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
734,AT-10215,2026-06-19,linkedin,UK,Investor,opinion piece,Another AI vendor saying trust us while public...,data_sovereignty,positive,NaN,947,0.79,NaN,Sovereign data explainer,possible duplicate,6,2026
737,AT-10685,2026-04-27,Reddit,UK,Academic,article,Critical infrastructure cannot rely on a black...,cyber_resilience,negative,NaN,558,0.75,trust correction,Security whitepaper,possible duplicate,4,2026
739,AT-10609,2026-05-09,X,UK,Critical infrastructure,press mention,Atlantic Tech is preparing to launch a civil d...,neutral_awareness,positive,15.0,1132,0.10,awareness,Brand awareness explainer,possible duplicate,5,2026
751,AT-10730,2026-05-09,linkedin,UK,Emergency services,influencer post,Public-sector AI procurement must be auditable...,procurement_trust,neutral,168.0,1876,0.04,pre-launch research,Transparent procurement one-pager,possible duplicate,5,2026


In [ ]:
sentiment_map = {
    "positive" : "positive",
    "pos" : "positive",
    "p" :"positive",
    "negative" : "negative",
    "neg" : "negative",
    "n" : "negative",
    "neu": "neutral",
    "neutral": "neutral"
}

clean_uk["sentiment_label"] = clean_uk["sentiment_label"].str.strip().str.lower()
clean_uk["sentiment_label"] = clean_uk["sentiment_label"].replace(sentiment_map)

In [ ]:
clean_uk.shape

(127, 17)

In [ ]:
clean_uk.tail()

,post_id,date,platform,country,audience_group,source_type,post_text,narrative_theme,sentiment_label,engagements,reach_estimate,misinformation_likelihood,campaign_phase,recommended_asset,notes,Calendar_Month,Calendar_Year
734,AT-10215,2026-06-19,linkedin,UK,Investor,opinion piece,Another AI vendor saying trust us while public...,data_sovereignty,positive,NaN,947,0.79,NaN,Sovereign data explainer,possible duplicate,6,2026
737,AT-10685,2026-04-27,Reddit,UK,Academic,article,Critical infrastructure cannot rely on a black...,cyber_resilience,negative,NaN,558,0.75,trust correction,Security whitepaper,possible duplicate,4,2026
739,AT-10609,2026-05-09,X,UK,Critical infrastructure,press mention,Atlantic Tech is preparing to launch a civil d...,neutral_awareness,positive,15.0,1132,0.10,awareness,Brand awareness explainer,possible duplicate,5,2026
751,AT-10730,2026-05-09,linkedin,UK,Emergency services,influencer post,Public-sector AI procurement must be auditable...,procurement_trust,neutral,168.0,1876,0.04,pre-launch research,Transparent procurement one-pager,possible duplicate,5,2026
756,NaN,2026-06-05,LinkedIn,UK,Government buyer,organic post,Faster procurement should not mean bypassing s...,procurement trust,negative,84.0,2450,0.67,trust correction,Transparent procurement one-pager,missing id,6,2026


In [ ]:
platform_map = {
    "x": "X/Twitter",
    "X": "X/Twitter",
    "twitter": "X/Twitter",
    "twitter/x": "X/Twitter",
    "Twitter/X": "X/Twitter",
    "x/twitter": "X/Twitter",
    "linkedin": "LinkedIn",
    "linked in": "LinkedIn",
    "yt": "YouTube",
    "YT": "YouTube",
    "youtube": "YouTube",
    "reddit": "Reddit",
    "Reddit": "Reddit",
    "Trade media": "Trade Media",
    "trade media": "Trade Media",
}

clean_uk["platform"] = clean_uk["platform"].replace(platform_map)


In [ ]:
narrative_map = {
    "cyber resilience":"cyber_resilience",
    "data sovereignty":"data_sovereignty",
    "dual use ethics":"dual_use_ethics",
    "positive innovation":"positive_innovation",
    "procurement trust":"procurement_trust"
}

clean_uk["narrative_theme"] = clean_uk["narrative_theme"].replace(narrative_map)
#clean_uk

1.1) Filling in missing values

In [ ]:
clean_uk = clean_uk.copy()
clean_uk["post_text"] = clean_uk["post_text"].fillna("Unknown")

#categorical fields
clean_uk["audience_group"] = clean_uk["audience_group"].fillna("Unknown")
clean_uk["source_type"] = clean_uk["source_type"].fillna("Unknown")
clean_uk["recommended_asset"] = clean_uk["recommended_asset"].fillna("Unknown")
clean_uk["narrative_theme"] = clean_uk["narrative_theme"].fillna("Unknown")

clean_uk = clean_uk.dropna(subset=["misinformation_likelihood"])

#smooth with median
numerical_columns = ["engagements", "reach_estimate"]
for cols in numerical_columns:
  clean_uk[cols] = pd.to_numeric(clean_uk[cols], errors="coerce")

for cols in numerical_columns:
  clean_uk[cols] = clean_uk[cols].fillna(clean_uk[cols].median())

clean_uk.shape

(121, 17)

In [ ]:
final_clean = clean_uk.copy()
final_clean = final_clean.reset_index(drop=True)
final_clean.tail()

,post_id,date,platform,country,audience_group,source_type,post_text,narrative_theme,sentiment_label,engagements,reach_estimate,misinformation_likelihood,campaign_phase,recommended_asset,notes,Calendar_Month,Calendar_Year
116,AT-10215,2026-06-19,LinkedIn,UK,Investor,opinion piece,Another AI vendor saying trust us while public...,data_sovereignty,positive,34.5,947.0,0.79,NaN,Sovereign data explainer,possible duplicate,6,2026
117,AT-10685,2026-04-27,Reddit,UK,Academic,article,Critical infrastructure cannot rely on a black...,cyber_resilience,negative,34.5,558.0,0.75,trust correction,Security whitepaper,possible duplicate,4,2026
118,AT-10609,2026-05-09,X/Twitter,UK,Critical infrastructure,press mention,Atlantic Tech is preparing to launch a civil d...,neutral_awareness,positive,15.0,1132.0,0.10,awareness,Brand awareness explainer,possible duplicate,5,2026
119,AT-10730,2026-05-09,LinkedIn,UK,Emergency services,influencer post,Public-sector AI procurement must be auditable...,procurement_trust,neutral,168.0,1876.0,0.04,pre-launch research,Transparent procurement one-pager,possible duplicate,5,2026
120,NaN,2026-06-05,LinkedIn,UK,Government buyer,organic post,Faster procurement should not mean bypassing s...,procurement_trust,negative,84.0,2450.0,0.67,trust correction,Transparent procurement one-pager,missing id,6,2026


2) Save csv

In [ ]:
#save csv
final_clean.to_csv("atlantic_tech_mock_social_listening_cleaned.csv", index=False)

3) Data analysis phase

questions I want to find out:

1) what is the overall sentiment
2) impact of social media platform on sentiments
3) marginalisation of data themes and sentiment
4) impact on audience group and sentiment
5) platforms that produce the most reach and the type of posts

In [ ]:
final_clean["sentiment_label"].value_counts(normalize = True)

,proportion
sentiment_label,
negative,0.553719
neutral,0.256198
positive,0.190083


In [ ]:
sentiment_platform = pd.crosstab(final_clean['sentiment_label'],final_clean['platform'],margins=True,normalize='columns') * 100
sentiment_platform = sentiment_platform.round(2)
sentiment_platform
# figure out which platforms to target to increase sentiment
# figure out which ones agree with sentiment

platform,Forum,LinkedIn,News comments,Policy blog,Reddit,Trade Media,X/Twitter,YouTube,All
sentiment_label,,,,,,,,,
negative,53.33,53.85,54.55,100.0,54.55,45.45,37.50,75.0,55.37
neutral,20.00,26.92,36.36,0.0,18.18,36.36,43.75,12.5,25.62
positive,26.67,19.23,9.09,0.0,27.27,18.18,18.75,12.5,19.01


In [ ]:
marginalise_themes = pd.crosstab(final_clean['sentiment_label'],final_clean['narrative_theme'],margins=True,normalize='columns') * 100
marginalise_themes
#figure out which topics to focus on and which ones alr produce positive sentiment

narrative_theme,cyber_resilience,data_sovereignty,dual_use_ethics,neutral_awareness,positive_innovation,procurement_trust,regulatory_compliance,All
sentiment_label,,,,,,,,
negative,68.181818,59.090909,81.25,14.285714,0.000000,60.0,85.714286,55.371901
neutral,22.727273,22.727273,18.75,28.571429,46.153846,35.0,7.142857,25.619835
positive,9.090909,18.181818,0.00,57.142857,53.846154,5.0,7.142857,19.008264


In [ ]:
audience_sentiment = pd.crosstab(final_clean['sentiment_label'],final_clean['audience_group'],margins=True,normalize='columns') * 100
audience_sentiment
#figure out sectors of audience and their sentiments

audience_group,Academic,Civil society,Critical infrastructure,Cybersecurity professional,Defence analyst,Emergency services,Government buyer,Investor,Journalist,Policy adviser,Public,Unknown,All
sentiment_label,,,,,,,,,,,,,
negative,62.50,50.000000,33.333333,50.0,55.555556,41.666667,70.0,42.857143,45.454545,69.230769,75.0,75.0,55.371901
neutral,18.75,28.571429,22.222222,37.5,33.333333,33.333333,30.0,14.285714,36.363636,7.692308,25.0,25.0,25.619835
positive,18.75,21.428571,44.444444,12.5,11.111111,25.000000,0.0,42.857143,18.181818,23.076923,0.0,0.0,19.008264


In [ ]:
filtered_sentiment = final_clean.copy()

platform_reach = (
    final_clean
    .groupby('platform')['reach_estimate']
    .agg(
        average_reach='mean',
        number_of_posts='count'
    )
    .reset_index()
    .sort_values(by='average_reach', ascending=False)
)

platform_reach
#which platform reaches the most people

,platform,average_reach,number_of_posts
7,YouTube,2395.187500,16
1,LinkedIn,2392.346154,26
6,X/Twitter,1528.687500,16
5,Trade Media,1368.090909,11
3,Policy blog,905.000000,4
4,Reddit,735.954545,22
2,News comments,550.363636,11
0,Forum,453.600000,15


Things understood by this exploration:
1) overall negative sentiment
2) policy blog and Youtube are the top 2 lowest positive sentiment performing
3) regulatory compliance and dual_use_ethics top 2 worst performers
4) Public are the most critical
5) Youtube is the platform with the highest overall reach regardless of positive/negative review